# Phase 4：把检索链路做成可用的 Mini RAG 产品

## 今天交付什么？

把前三个阶段组合成一个别人可以启动、查询、查看证据的服务：

```text
文档目录 -> KnowledgeBase -> FastAPI -> /search 与 /chat -> 引用结果
```

本项目选择 **evidence-first**：没有 LLM API Key 时，`/search` 仍然可用，`/chat` 返回带 Chunk ID 的证据模式答案；有明确配置时才允许调用可选 LLM。这不是功能缩水，而是把“证据能否被检索”与“语言模型如何表达”分开，便于开发、测试和追责。

**完成后你要能回答：**

1. API 为什么要有清晰的请求/响应合同？
2. 为什么引用字段是产品核心，而不是调试信息？
3. 为什么模型和索引应该在服务启动时复用，而不是每次请求重建？
4. 没有 LLM Key 时，系统如何安全地降级？

## Evidence Quest 任务卡：Phase 4 总览：上线证据工作台

**你的身份：** Mini RAG 发布负责人  
**案件背景：** 最后一公里是让真实用户能查询、能看到引用、遇到无证据时能得到诚实回答。

### 本关专业 Goal

把解析、检索、评估和 API 组合成可验收的 Mini RAG 产品。

### 你要交付的作品

**可启动、可测试、可追溯的 Evidence Desk**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：Evidence Desk 发布官  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 0. 前置知识与完成定义

| 知识 | 今天的用法 | 通过标准 |
| --- | --- | --- |
| HTTP 方法/状态码 | GET 健康检查、POST 查询 | 能解释 200 和 422 |
| JSON | 传输 Query、结果和 citations | 能定位结果字段 |
| Pydantic | 限制空 Query 和非法 top_k | 能触发并解释校验错误 |
| TestClient | 不启动外部服务器也测试 API | 能写出契约断言 |
| 环境变量 | 控制可选 LLM，避免密钥入库 | 能说明默认安全行为 |

**Definition of Done：** 新用户只看 README 就能启动；提交一个问题得到结果；结果包含 source/page/chunk_id；没有 API Key 也不会偷偷产生外部调用。

In [1]:
from pathlib import Path
import json
import sys


def find_project_root() -> Path:
    """兼容从项目根目录、notebooks 目录或 JupyterLab 启动目录运行。"""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目根目录:", ROOT)

from fastapi.testclient import TestClient
from phase4_mini_rag_system.app import create_app

app = create_app(ROOT / "phase1_doc_parser" / "examples" / "input")
client = TestClient(app)
print("FastAPI app 已创建，测试不会启动独立端口。")

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


FastAPI app 已创建，测试不会启动独立端口。


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase4'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase4
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


## 1. `/health`：先确认服务准备好了

健康检查不是装饰页面。它告诉调用方：服务是否能响应、当前索引里有多少 Chunk、索引版本是什么。没有 `index_version` 时，用户很难判断“我刚刚导入的文档是否真的生效”。

In [3]:
health = client.get("/health")
print("status:", health.status_code)
print(json.dumps(health.json(), ensure_ascii=False, indent=2))
assert health.status_code == 200
assert health.json()["chunks"] > 0
assert health.json()["index_version"].startswith("chunks-")

status: 200
{
  "status": "ok",
  "chunks": 2,
  "index_version": "chunks-2-size-512-overlap-128"
}


## 2. `/search`：先建立证据合同

搜索接口返回的不只是字符串列表。每个结果都要能回答：

- 这段证据的稳定 ID 是什么？
- 来自哪个文件、哪一页？
- 为什么排在这里？（至少保留可比较的 score）
- 当前索引是哪一版？

这是“可解释检索”的最小合同，也是未来生成答案时允许模型使用的证据边界。

In [4]:
search_response = client.post("/search", json={"query": "Chunk overlap", "top_k": 5})
payload = search_response.json()
print("status:", search_response.status_code)
print(json.dumps(payload, ensure_ascii=False, indent=2))

assert search_response.status_code == 200
assert payload["results"]
required_citation_fields = {"chunk_id", "text", "source", "page", "score"}
assert required_citation_fields <= payload["results"][0].keys()

status: 200
{
  "query": "Chunk overlap",
  "results": [
    {
      "chunk_id": "656e2c0615cc360f",
      "text": "# Phase 1 Quickstart\n\n文档解析的目标不是尽快删除格式信息，而是保留足够的来源元数据，让后续检索结果可以回溯到原文。\n\n## Chunk 策略\n\n先按段落和换行切分，再按中文标点递归降级。overlap 用于保留跨边界的上下文，但会增加索引体积和重复召回。",
      "source": "D:\\code\\codeByCursor\\AI_EXAM\\ai-search-rag-internship\\phase1_doc_parser\\examples\\input\\quickstart.md",
      "page": null,
      "score": 1.318776,
      "metadata": {
        "source": "D:\\code\\codeByCursor\\AI_EXAM\\ai-search-rag-internship\\phase1_doc_parser\\examples\\input\\quickstart.md",
        "page": null,
        "chunk_index": 0,
        "metadata": {
          "format": "markdown",
          "headings": [
            "Phase 1 Quickstart",
            "Chunk 策略"
          ]
        }
      }
    }
  ],
  "trace_id": "search-652091869fc0",
  "index_version": "chunks-2-size-512-overlap-128"
}


### “有引用”不等于“引用支持答案”

系统返回一个 source 只能证明“它来自某处”，不能自动证明它支持回答。更严格的产品验收还要检查：答案中的关键事实是否能在引用 Chunk 中找到。当前 evidence-only 模式直接展示证据，减少生成层把引用和事实脱钩的机会。

In [5]:
citation = payload["results"][0]
print({
    "citation_id": citation["chunk_id"],
    "source": citation["source"],
    "page": citation["page"],
    "score": citation["score"],
    "text_preview": citation["text"][:120],
})

{'citation_id': '656e2c0615cc360f', 'source': 'D:\\code\\codeByCursor\\AI_EXAM\\ai-search-rag-internship\\phase1_doc_parser\\examples\\input\\quickstart.md', 'page': None, 'score': 1.318776, 'text_preview': '# Phase 1 Quickstart\n\n文档解析的目标不是尽快删除格式信息，而是保留足够的来源元数据，让后续检索结果可以回溯到原文。\n\n## Chunk 策略\n\n先按段落和换行切分，再按中文标点递归降级。overlap 用于保留跨边界的'}


## 3. `/chat`：生成不是检索的替代品

`/chat` 的内部顺序是：

1. 用同一个 KnowledgeBase 检索。
2. 将结果作为受限证据传给回答层。
3. 返回答案、模式和 citations。

当前没有显式开启 `RAG_ENABLE_LLM=true` 且没有 `OPENAI_API_KEY`，所以应该得到 `evidence-only`。这让 Notebook 可离线运行，也防止学习时不小心产生 API 费用。

In [6]:
chat_response = client.post("/chat", json={"query": "Chunk overlap", "top_k": 3})
chat_payload = chat_response.json()
print("status:", chat_response.status_code)
print("mode:", chat_payload["mode"])
print("answer:\n", chat_payload["answer"])
print("citations:", len(chat_payload["citations"]))
assert chat_response.status_code == 200
assert chat_payload["mode"] in {"evidence-only", "evidence-only-fallback", "llm"}
assert chat_payload["citations"]

status: 200
mode: evidence-only
answer:
 当前为 evidence-only 模式。请依据以下可追溯证据作答：
[656e2c0615cc360f] # Phase 1 Quickstart

文档解析的目标不是尽快删除格式信息，而是保留足够的来源元数据，让后续检索结果可以回溯到原文。

## Chunk 策略

先按段落和换行切分，再按中文标点递归降级。overlap 用于保留跨边界的上下文，但会增加索引体积和重复召回。
citations: 1


## 4. 错误处理：让调用方知道问题在输入还是系统

用户输入错误和服务器故障不能都返回 500：

- 空 Query：请求不符合 schema，FastAPI 返回 422。
- 不存在的 source：这是合法查询，但结果可以为空。
- 导入不存在目录：返回 400，并给出稳定错误码。

错误合同越清晰，前端和自动化测试越容易正确处理，也越容易定位问题。

In [7]:
empty_query = client.post("/search", json={"query": ""})
unknown_source = client.post("/search", json={"query": "overlap", "source": "does-not-exist.md"})
bad_ingest = client.post("/documents/ingest", json={"input_dir": str(ROOT / "missing-input")})

print("empty query:", empty_query.status_code, empty_query.json().get("detail"))
print("unknown source:", unknown_source.status_code, unknown_source.json()["results"])
print("bad ingest:", bad_ingest.status_code, bad_ingest.json())
assert empty_query.status_code == 422
assert unknown_source.status_code == 200 and unknown_source.json()["results"] == []
assert bad_ingest.status_code == 400

empty query: 422 [{'type': 'string_too_short', 'loc': ['body', 'query'], 'msg': 'String should have at least 1 character', 'input': '', 'ctx': {'min_length': 1}}]
unknown source: 200 []
bad ingest: 400 {'detail': {'code': 'INPUT_DIR_NOT_FOUND', 'message': 'Input directory does not exist: D:\\code\\codeByCursor\\AI_EXAM\\ai-search-rag-internship\\missing-input'}}


## 5. TestClient：把用户故事变成可执行验收

用户故事：

> 作为需要查阅技术资料的实习生，我输入一个问题，希望看到答案和能回到原文的证据；当没有足够证据时，系统应该明确说不知道，而不是编造。

对应测试至少覆盖：健康、检索引用、无 LLM 降级、空输入、无结果和导入错误。Notebook 里的断言是快速学习反馈，`tests/` 里的 pytest 才是提交前的长期保护。

In [8]:
def assert_search_contract(query: str) -> dict:
    response = client.post("/search", json={"query": query, "top_k": 3})
    assert response.status_code == 200
    body = response.json()
    assert {"query", "results", "trace_id", "index_version"} <= body.keys()
    return body

contract = assert_search_contract("Dense BM25")
print("契约通过，trace_id:", contract["trace_id"])
print("结果数:", len(contract["results"]))

契约通过，trace_id: search-515b3f2b38f6
结果数: 1


## 6. 真实产品交付：把演示结果保存为证据

不要只在屏幕上看一眼结果。把一次 Demo 的配置、问题、回答模式、引用和索引版本保存下来，技术报告才能复盘，面试时也能展示系统确实运行过。

In [9]:
demo_record = {
    "project": "EvidenceDesk Mini RAG",
    "index_version": chat_payload["index_version"],
    "query": chat_payload["query"],
    "mode": chat_payload["mode"],
    "answer": chat_payload["answer"],
    "citations": [
        {key: item.get(key) for key in ("chunk_id", "source", "page", "score")}
        for item in chat_payload["citations"]
    ],
}
demo_path = ROOT / "data" / "processed" / "phase4_demo_record.json"
demo_path.write_text(json.dumps(demo_record, ensure_ascii=False, indent=2), encoding="utf-8")
print("Demo 记录已保存:", demo_path)

Demo 记录已保存: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\phase4_demo_record.json


## 最终项目验收

完成后，从项目根目录启动：

```powershell
conda activate 'F:\anaconda\miniconda3\envs\ai-rag-internship'
python -m phase4_mini_rag_system
```

浏览器打开 `http://127.0.0.1:8000/`，完成至少三种演示：

1. 能回答的问题：显示回答和引用。
2. 同义表达：观察 BM25 baseline 的能力边界，并记录下一步 Dense 实验。
3. 不可回答的问题：系统明确返回“没有足够证据”，不编造。

**最终系统链路：** 解析 → 分块 → 稳定 ID/来源 → BM25 检索 → FastAPI → evidence-only/可选 LLM → citations。

**最终交付清单：** 代码、4 个可运行 Notebook、测试、README、前置知识矩阵、实验记录、PRD/技术报告素材，以及 `phase4_demo_record.json`。

## Boss Challenge：从用户 Query 开始展示完整链路，并证明无证据时系统不会编造。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [10]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [11]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/phase4_acceptance_record.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\phase4_acceptance_record.json']
